# Kerangka chatbot


## 1. Persiapan folder proyek dan modul

In [1]:
import os
import sys
import json
from pathlib import Path

# Cari root proyek 
file_wajib = {
    "question_processing.py",
    "passage_retrieval.py",
    "answer_processing.py",
    "validasi_tokoh.py",
    "korpus_kalimat_final.json",
}
folder_aktif = Path.cwd().resolve()
folder_proyek = next(
    (p for p in [folder_aktif, *folder_aktif.parents]
     if all((p / nama).is_file() for nama in file_wajib)),
    None,
)
if folder_proyek is None:
    raise FileNotFoundError(
        "Buka folder pba-qa-sejarah-indonesia di VS Code, lalu jalankan notebook ini."
    )

os.chdir(folder_proyek)  # passage_retrieval.py memuat korpus dengan path relatif
if str(folder_proyek) not in sys.path:
    sys.path.insert(0, str(folder_proyek))

from question_processing import process_question
from passage_retrieval import cari
from answer_processing import ambil_jawaban as ekstrak_jawaban

K = 20
print(f"Folder proyek: {folder_proyek}")
print(f"Modul Question Processing, BM25, dan Answer Processing siap | k={K}")

Folder proyek: C:\Users\Shatara Belva M\Videos\pba-qa-sejarah-indonesia
Modul Question Processing, BM25, dan Answer Processing siap | k=20


## 2. Fungsi chatbot

- muat_data
- proses_pertanyaan
- cari_kandidat
- ambil_jawaban
- jawab_pertanyaan dan main



In [ ]:
def muat_data(nama_file="korpus_kalimat_final.json"):
    with open(nama_file, encoding="utf-8") as f:
        data = json.load(f)
    return data["kalimat"], data["metadata"]["aturan_alias"]


def proses_pertanyaan(pertanyaan, alias):
    return process_question(pertanyaan, alias)


def cari_kandidat(info_pertanyaan, korpus, k=K):
    return cari(info_pertanyaan["normalized_question"], k=k)


def ambil_jawaban(info_pertanyaan, kandidat, kalimat_korpus=None):
    return ekstrak_jawaban(
        info_pertanyaan,
        kandidat,
        kalimat_korpus=kalimat_korpus,
    )


def jawab_pertanyaan(pertanyaan, korpus, alias, k=K):
    info = proses_pertanyaan(pertanyaan, alias)
    kandidat = cari_kandidat(info, korpus, k=k)
    if not kandidat:
        return None
    return ambil_jawaban(info, kandidat, kalimat_korpus=korpus)


def main(nama_file="korpus_kalimat_final.json", k=K):
    korpus, alias = muat_data(nama_file)
    print("Chatbot QA Sejarah Indonesia")
    print("Jumlah kalimat:", len(korpus))
    print(f"BM25 k={k} | Ketik 'keluar' untuk berhenti.")
    while True:
        try:
            pertanyaan = input("\nKamu: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSelesai.")
            break
        if pertanyaan.casefold() in {"keluar", "exit", "quit"}:
            print("Selesai.")
            break
        if not pertanyaan:
            continue
        hasil = jawab_pertanyaan(pertanyaan, korpus, alias, k=k)
        if hasil and hasil.get("jawaban"):
            print("Bot:", hasil["jawaban"])
            print("ID bukti:", hasil.get("id_kalimat"))
        else:
            print("Bot: Jawaban belum ditemukan dalam korpus.")

## 3. Uji 1 pertanyaan

In [3]:
korpus, alias = muat_data()
contoh = "Kapan Mohammad Hatta lahir?"
hasil = jawab_pertanyaan(contoh, korpus, alias)
print("Pertanyaan:", contoh)
print("Jawaban:", hasil["jawaban"] if hasil else "Tidak ditemukan")
print("ID bukti:", hasil.get("id_kalimat") if hasil else "-")

Pertanyaan: Kapan Mohammad Hatta lahir?
Jawaban: 12 Agustus 1902
ID bukti: D08-B0004-K003


## 4. Chat

In [5]:
JALANKAN_CHAT = True

if JALANKAN_CHAT:
    main(k=K)
else:
    print("Siap dipakai. Ubah JALANKAN_CHAT = True untuk membuka percakapan.")

Chatbot QA Sejarah Indonesia
Jumlah kalimat: 3140
BM25 k=20 | Ketik 'keluar' untuk berhenti.
Bot: Mohammad Hatta
ID bukti: D03-B0001-K002
Bot: 12 Agustus 1902
ID bukti: D08-B0004-K003
Selesai.
